In [0]:
%run ../utils/config_loader

In [0]:
from pyspark.sql.functions import row_number, lit
from pyspark.sql.window import Window

config = load_config('dev')
silver_orders = spark.table(f"{config['catalog']}.{config['schemas']['silver']}.orders_clean")
dim_customers = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimCustomers")
dim_products = spark.table(f"{config['catalog']}.{config['schemas']['gold']}.DimProducts")

fact_orders = (silver_orders
    .join(dim_customers, silver_orders.customer_id == dim_customers.customer_id, "left")
    .join(dim_products, silver_orders.product_id == dim_products.product_id, "left")
    .withColumn("FactOrderKey", row_number().over(Window.orderBy(lit(1))))
    .select("FactOrderKey", "order_id", dim_customers.DimCustomerKey, dim_products.DimProductKey, "order_date", "quantity", "total_amount")
)

fact_table = f"{config['catalog']}.{config['schemas']['gold']}.FactOrders"
fact_orders.write.format("delta").mode("overwrite").saveAsTable(fact_table)

print(f"✅ {fact_table}: {spark.table(fact_table).count()} records")